In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("qoute_dataset.csv")

In [4]:
df

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
...,...,...
3033,The past beats inside me like a second heart.,"John Banville,"
3034,"Damn, Claire. Warn a guy before you do a face-...","Rachel Caine,"
3035,"Can you be a girl for a few seconds?""""I'm alwa...","Veronica Roth,"
3036,That's what fiction is for. It's for getting a...,Tim O'Brien


In [5]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [6]:
quotes = quotes.str.lower()

In [7]:
import string
translator = str.maketrans('','', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [8]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


# **Tokenization**

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
vocab_size = 10000
tokeinizer = Tokenizer(num_words=vocab_size)
tokeinizer.fit_on_texts(quotes)

In [11]:
word_index = tokeinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokeinizer.texts_to_sequences(quotes)

In [13]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [14]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [15]:
X = []
y = []

for seq in sequence:
  for i in range(1, len(seq)): # Changed from len(sequence) to len(seq)
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [16]:
len(X)

85271

In [17]:
len(y)

85271

In [18]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [21]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [22]:
y = np.array(y)

In [23]:
X_padded.shape

(85271, 745)

In [24]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [25]:
y_one_hot.shape

(85271, 10000)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN ,LSTM, Dense

In [27]:
embedding_dim = 50
rnn_units = 128

In [28]:
rnn_model = Sequential()

rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [29]:
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [30]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
lstm_model = Sequential()

lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [32]:
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [33]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
epochs = 10
batch_size = 128

In [35]:
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 70ms/step - accuracy: 0.0444 - loss: 6.7158 - val_accuracy: 0.0565 - val_loss: 6.5600
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0764 - loss: 6.1298 - val_accuracy: 0.0848 - val_loss: 6.3346
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0981 - loss: 5.7828 - val_accuracy: 0.1008 - val_loss: 6.3092
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1139 - loss: 5.5037 - val_accuracy: 0.1033 - val_loss: 6.3163
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1274 - loss: 5.2540 - val_accuracy: 0.1089 - val_loss: 6.3758
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1388 - loss: 5.0233 - val_accuracy: 0.1110 - val_loss: 6.4556
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1523 - loss: 4.8079 - val_accuracy: 0.1068 - val_loss: 6.4837
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1687 - loss: 4.6039 - 

In [36]:
epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 56ms/step - accuracy: 0.0386 - loss: 6.7596 - val_accuracy: 0.0473 - val_loss: 6.6804
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0572 - loss: 6.3221 - val_accuracy: 0.0631 - val_loss: 6.5685
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0783 - loss: 6.0605 - val_accuracy: 0.0841 - val_loss: 6.4774
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0960 - loss: 5.8418 - val_accuracy: 0.0936 - val_loss: 6.4567
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1063 - loss: 5.6683 - val_accuracy: 0.1021 - val_loss: 6.4390
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1177 - loss: 5.5032 - val_accuracy: 0.1080 - val_loss: 6.4357
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1265 - loss: 5.3473 - val_accuracy: 0.1081 - val_loss: 6.4606
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1343 - loss: 5

In [37]:
lstm_model.save('lstm_model.h5')

In [38]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [44]:
from re import VERBOSE
from IPython.core.displayhook import tokenize
def predictor(model, tokeinizer, text, max_len):
  text = text.lower()
  seq = tokeinizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')
  pred = model.predict(seq, verbose=0)
  pred_word_index = np.argmax(pred)
  return index_to_word[pred_word_index]

In [49]:
seed_text = "life is about"
next_word = predictor(lstm_model, tokeinizer, seed_text, max_len)
print(f"Next Word: {next_word}")

Next Word: accepting


In [55]:
def generate_text(model, tokenizer, seed_text, max_len, num_words):
  for _ in range(num_words):
    next_word = predictor(model, tokenizer, seed_text, max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [56]:
seed = "life is about"
generate_text = generate_text(lstm_model, tokeinizer, seed, max_len, 10)
print(generate_text)

life is about accepting the challenges along the way choosing to keep moving


In [66]:
import pickle
with open('tokenizer.pkl', 'wb') as file:
  pickle.dump(tokeinizer, file)

In [67]:
with open('max_len', 'wb') as file:
  pickle.dump(max_len, file)

In [69]:
# max_len is an integer, so we use pickle to save it (as done in the previous cell)
import pickle
with open('max_len.pkl', 'wb') as file:
    pickle.dump(max_len, file)
print("max_len saved to max_len.pkl")

max_len saved to max_len.pkl
